In [ ]:
import random
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import copy
import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely.geometry import LineString, Point
from shapely.ops import split
import folium
import pyproj
import osmnx as ox
from tqdm import tqdm
import igraph as ig
from scipy.spatial import cKDTree
import json
import re
import os

In [ ]:
# map grid
drt_grid=gpd.read_file('Data/DRT Grid (Buffer).gpkg')

# depot
depot=pd.read_csv('Data/depot.csv')
depot=drt_grid.merge(depot).reset_index(drop=True)

# create the network graph
edges=pd.read_csv('E:/On-demand Transport Service/Analysis/Data/Road Network/Network_Yilan.csv')
edges=edges.merge(pd.read_csv('E:/On-demand Transport Service/Analysis/Data/Road Network/osm_weight.csv').rename(columns={'way': 'highway'}))
edges['TravelTime']=edges.d/1000/(edges.max_speed*edges.adjustment)*3600

nodes=pd.concat([edges[['from_id','from_lon','from_lat']].rename(columns={'from_id':'node_id', 'from_lon':'x', 'from_lat':'y'}),
                 edges[['to_id','to_lon','to_lat']].rename(columns={'to_id':'node_id', 'to_lon':'x', 'to_lat':'y'})], axis=0).drop_duplicates().reset_index(drop=True)
nodes['node_id_new']=nodes.index
kdtree=cKDTree(np.column_stack([nodes['x'].values, nodes['y'].values]))

edges_dt=edges.copy()
edges_dt=edges_dt.merge(nodes.loc[:,['node_id','node_id_new']], left_on='from_id', right_on='node_id').drop(columns=['node_id']).rename(columns={'node_id_new':'from_id_new'})
edges_dt=edges_dt.merge(nodes.loc[:,['node_id','node_id_new']], left_on='to_id', right_on='node_id').drop(columns=['node_id']).rename(columns={'node_id_new':'to_id_new'})

g=ig.Graph(
    n=len(nodes),
    edges=list(zip(edges_dt['from_id_new'], edges_dt['to_id_new'])),
    directed=True
)
g.es['TravelTime']=edges_dt['TravelTime'].astype(float).tolist()

edges_sf=edges_dt.copy()
edges_sf=edges_sf.drop(columns=['from_id','to_id']).rename(columns={'from_id_new':'from_id','to_id_new':'to_id'})
edges_sf['geometry'] = edges_sf.apply(lambda r: LineString([(r.from_lon, r.from_lat), (r.to_lon, r.to_lat)]), axis=1)
edges_sf=gpd.GeoDataFrame(edges_sf, geometry='geometry', crs='EPSG:4326')
edges_sf['Distance']=edges_sf.to_crs(8058).geometry.length

# Label each node with its component ID
components = g.clusters(mode='strong') # or 'weak' depending on your graph type
g.vs['component'] = components.membership

# Identify the largest component (the "main" network)
main_component_id = pd.Series(components.membership).value_counts().idxmax()

# Get indices of nodes that are actually in the main network
valid_node_indices = [v.index for v in g.vs if v['component'] == main_component_id]
valid_nodes_df = nodes.iloc[valid_node_indices]
kdtree_valid=cKDTree(np.column_stack([valid_nodes_df['x'].values, valid_nodes_df['y'].values]))

In [ ]:
def nearest_node(x, y):
    # This query now only considers nodes that are part of the main network
    dist, idx_in_tree = kdtree_valid.query([x, y])
    
    # Map the tree index back to the actual graph node index
    actual_node_id = valid_nodes_df.index[idx_in_tree]
    return actual_node_id


def travel_time(X_O, Y_O, X_D, Y_D):
    o_id=nearest_node(X_O, Y_O)
    d_id=nearest_node(X_D, Y_D)

    path_time=g.distances(source=o_id, target=d_id, weights='TravelTime')[0][0]
    return(path_time)


def travel_path(X_O, Y_O, X_D, Y_D):
    o_id=nearest_node(X_O, Y_O)
    d_id=nearest_node(X_D, Y_D)

    path=g.get_shortest_paths(o_id, d_id, weights='TravelTime')[0]
    path_time=g.distances(source=o_id, target=d_id, weights='TravelTime')[0][0]
#     path_geo=gpd.GeoDataFrame(pd.DataFrame({'from_id': path[:-1], 'to_id': path[1:]}).merge(edges_sf))
#     m=folium.Map(location=[paths["from_lat"].mean(), path["from_lon"].mean()], zoom_start=14, tiles='CartoDB positron_nolabels')
#     folium.GeoJson(path_geo).add_to(m)
#     folium.LayerControl().add_to(m)
#     m
#     path_time=sum(G[u][v]['weight'] for u, v in zip(path[:-1], path[1:]))
    return({'path':path, 'traveltime':path_time})


def edge_cost(a, b):
    if a is None or b is None:
        return 0
    return travel_time(a[0], a[1], b[0], b[1])


def find_best_insertion(pos, current_route, pax_new):
    route_ids = list(current_route.keys())
    route_pts = list(current_route.values())

    n = len(route_pts)

    pickup_id = "O_" + str(pax_new["id"])
    dropoff_id = "D_" + str(pax_new["id"])

    O = pax_new["origin"]
    D = pax_new["dest"]

    min_added_cost = float("inf")
    best_i = None
    best_j = None

    # i = pickup insertion position
    for i in range(n + 1):

        # node before pickup
        A = pos if i == 0 else route_pts[i - 1]

        # node after pickup
        B = route_pts[i] if i < n else None

        # cost change caused by inserting pickup
        pickup_delta = (
            edge_cost(A, O)
            + edge_cost(O, B)
            - edge_cost(A, B)
        )

        # j = dropoff insertion position after pickup
        for j in range(i + 1, n + 2):

            if j == i + 1:
                # pickup and dropoff are consecutive
                # A -> B becomes A -> O -> D -> B
                added_cost = (
                    edge_cost(A, O)
                    + edge_cost(O, D)
                    + edge_cost(D, B)
                    - edge_cost(A, B)
                )

            else:
                # dropoff is inserted later
                # after inserting pickup, dropoff index j corresponds to
                # original edge route_pts[j-2] -> route_pts[j-1]
                C = route_pts[j - 2]
                E = route_pts[j - 1] if (j - 1) < n else None

                dropoff_delta = (
                    edge_cost(C, D)
                    + edge_cost(D, E)
                    - edge_cost(C, E)
                )

                added_cost = pickup_delta + dropoff_delta

            if added_cost < min_added_cost:
                min_added_cost = added_cost
                best_i = i
                best_j = j
                
    best_route=current_route.copy()
    best_route_key=list(best_route.keys())
    best_route_value=list(best_route.values())

    best_route_key.insert(best_i, 'O_'+str(pax_new['id']))
    best_route_value.insert(best_i, pax_new['origin'])
    best_route_key.insert(best_j, 'D_'+str(pax_new['id']))
    best_route_value.insert(best_j, pax_new['dest'])
    best_route=dict(zip(best_route_key, best_route_value))

    return {
        "added_cost": min_added_cost,
        "route": best_route
    }

            
def get_route_time_edge(start_pos, waypoints, current_time):
    if not waypoints:
        return 0
    
    total_time=0
    current=start_pos
    path_route=[]
    path_od=[]
    for p in waypoints:
        path_info=travel_path(current[0], current[1], waypoints[p][0], waypoints[p][1])
        total_time += path_info['traveltime']
        
        temp=path_info['path']
        if len(temp)==1:
            temp.append(temp[0])
        path_route.append(temp)
        
        path_od.append([None]*(len(temp)-2) + [p])
        current=waypoints[p]
        
    path_edge=pd.DataFrame([(route_id, o, d) for route_id, r in enumerate(path_route) for o, d in zip(r[:-1], r[1:])], columns=['route','from_id','to_id']).merge(edges_sf.loc[:,['edge_id','from_id','to_id','TravelTime','Distance','geometry']], how='left')
    path_edge.loc[path_edge.from_id==path_edge.to_id, 'TravelTime']=0
    path_edge['path_od']=np.concatenate(path_od)
    path_time=dict(zip(list(waypoints.keys()), list(path_edge.groupby('route').agg({'TravelTime':'sum'}).TravelTime)))
    predict_arr=dict(pd.Series(path_time).cumsum()/60+current_time) # in minutes
    return dict({'total_time':total_time, 'path_time':path_time, 'predict_arr':predict_arr, 'path_edge':path_edge})


def get_route_time(start_pos, waypoints, current_time):
    if not waypoints:
        return {"total_time": 0, "path_time": {}, "predict_arr": {}, "path_info":{}}

    total_time = 0
    current = start_pos
    path_time = {}

    path_edge_all=[]
    for p, coord in waypoints.items():
        path_edge = travel_path(current[0], current[1], coord[0], coord[1])
        tt = path_edge["traveltime"]
        total_time += tt
        path_time[p] = tt

        path_edge_all.append(path_edge)
        current = coord

    predict_arr = (
        pd.Series(path_time).cumsum() / 60 + current_time
    ).to_dict()

    return {
        "total_time": total_time,
        "path_time": path_time,
        "predict_arr": predict_arr,
        "path_edge_all": path_edge_all
    }


def get_route_edge(waypoints, path_info_all):
    if not waypoints:
        return None
    
    path_route=[]
    path_od=[]
    for p in range(len(waypoints)):
        temp=path_info_all[p]['path']
        if len(temp)==1:
            temp.append(temp[0])
        path_route.append(temp)
        
        path_od.append([None]*(len(temp)-2) + [list(waypoints)[p]])
        
    path_edge=pd.DataFrame([(route_id, o, d) for route_id, r in enumerate(path_route) for o, d in zip(r[:-1], r[1:])], columns=['route','from_id','to_id']).merge(edges_sf.loc[:,['edge_id','from_id','to_id','TravelTime','Distance','geometry']], how='left')
    path_edge.loc[path_edge.from_id==path_edge.to_id, 'TravelTime']=0
    path_edge['path_od']=np.concatenate(path_od)
    path_edge['DistTravel']=[0.0]*len(path_edge)
    return path_edge

In [ ]:
def total_existing_delay(old_arr, new_arr):
    delay_sum = 0

    for stop_id in old_arr.keys():
        if stop_id.startswith("D_"):
            delay_sum += max(0, new_arr[stop_id] - old_arr[stop_id])

    return delay_sum

def find_best_insertion_with_penalty(
    pos,
    current_route,
    current_time,
    pax_new,
    alpha=1.5,   # penalty for delaying existing passengers
    beta=3.0,    # penalty for new passenger waiting time
#     alpha=3,   # penalty for delaying existing passengers
#     beta=1.5,    # penalty for new passenger waiting time
    max_existing_delay=None,
    max_new_wait=None
):

    pickup_id = "O_" + str(pax_new["id"])
    dropoff_id = "D_" + str(pax_new["id"])

    O = pax_new["origin"]
    D = pax_new["dest"]

    min_total_cost = float("inf")
    best_route = None
    best_info = None
    
    if len(current_route)==0:            
        temp_route=dict(zip(['O_'+str(pax_new['id']), 'D_'+str(pax_new['id'])], [pax_new['origin'],pax_new['dest']]))
        new_info=get_route_time(pos, temp_route, current_time)
        new_arr = new_info["predict_arr"]
        added_vehicle_time=new_info['total_time']
        new_wait_time = (new_arr[pickup_id]-current_time)*60
        total_cost = (
                added_vehicle_time
                + beta * new_wait_time
            )
        if total_cost < min_total_cost:
            min_total_cost = total_cost
            best_route = temp_route
            best_info = new_info
            best_added_vehicle_time=added_vehicle_time

    else:
        route_ids = list(current_route.keys())
        route_pts = list(current_route.values())

        n = len(route_pts)
        # Current route information
        current_info = get_route_time(pos, current_route, current_time)
        current_arr = current_info["predict_arr"]
        current_total_time = current_info["total_time"]
    
        for i in range(n + 1):
            for j in range(i + 1, n + 2):

                temp_keys = route_ids[:]
                temp_vals = route_pts[:]

                temp_keys.insert(i, pickup_id)
                temp_vals.insert(i, O)

                temp_keys.insert(j, dropoff_id)
                temp_vals.insert(j, D)

                temp_route = dict(zip(temp_keys, temp_vals))

                # Candidate route information
                new_info = get_route_time(pos, temp_route, current_time)
                new_arr = new_info["predict_arr"]
                new_total_time = new_info["total_time"]

                # 1. Added vehicle travel time
                added_vehicle_time = new_total_time - current_total_time

                # 2. Existing passenger delay penalty
                existing_delay = 0

                for stop_id in current_arr.keys():
                    if stop_id.startswith("D_"):
                        delay = (new_arr[stop_id] - current_arr[stop_id])*60
                        delay = max(0, delay)
                        existing_delay += delay

                        if max_existing_delay is not None and delay > max_existing_delay: # do not allow delay over a threshold
                            existing_delay = float("inf")
                            break

                if existing_delay == float("inf"):
                    continue

                # 3. New passenger waiting time
                new_wait_time = (new_arr[pickup_id]-current_time)*60

                if max_new_wait is not None and new_wait_time > max_new_wait:
                    continue

                # 4. Penalised objective
                total_cost = (
                    added_vehicle_time
                    + alpha * existing_delay
                    + beta * new_wait_time
                )

                if total_cost < min_total_cost:
                    min_total_cost = total_cost
                    best_route = temp_route
                    best_info = new_info
                    best_added_vehicle_time=added_vehicle_time

    if best_route is None:
        return None

    return {
        "total_cost": min_total_cost,
        "added_vehicle_time": best_added_vehicle_time,
#         "existing_passenger_delay": total_existing_delay(current_arr, best_info["predict_arr"]),
#         "new_passenger_wait_time": best_info["predict_arr"][pickup_id],
        "route": best_route,
        "edge_info": best_info
    }


def find_best_vehicle(pax_new, vehicles, current_time, alpha=1.5, beta=3.0):
    total_cost=[]
    all_res=[]
    for v in vehicles:
        temp=find_best_insertion_with_penalty(v.pos, v.waypoints, current_time, pax_new, alpha, beta)
        total_cost.append(temp['total_cost'])
        all_res.append(temp)
    
    veh_id=np.argmin(total_cost)
    return(dict({'veh_id':int(veh_id), 'result':all_res[veh_id]}))

In [ ]:
SIM_STEPS = 30  # report every 30 secs

class AgentVehicle:
    def __init__(self, id, seat_capacity):
        self.id = id
        
        temp=depot.iloc[random.sample(range(len(depot)), k=1)].reset_index(drop=True).geometry.sample_points(size=1)[0]
        self.pos = [temp.x, temp.y] # depot
        
        self.edge_pass_per = 0
        self.seat_capacity = seat_capacity
        self.pass_od=[]
        self.waypoints = [] 
        self.path_info = [] 
        self.assign_pax = [] 

    def move(self):
        if not self.waypoints:
            return 
        
        route_temp=self.path_info['path_edge'].copy()
        if sum(route_temp.TravelTime)<SIM_STEPS:
            temp=route_temp.geometry
            temp=temp[~temp.isna()]
            temp=list(temp[len(temp)-1].coords)[-1]
            
            self.pos=[temp[0], temp[1]]
            self.edge_pass_per=0
#             self.waypoints = []
            
            temp=route_temp.path_od
            self.pass_od=list(temp[~temp.isna()])
            
            self.path_info = []
            return
        
        tt_cum=np.cumsum(route_temp.TravelTime)
        loc_edge=(tt_cum>SIM_STEPS).idxmax()
        
        if loc_edge!=0:
            pass_per_new=(SIM_STEPS-tt_cum[(loc_edge-1)])/(route_temp.TravelTime[loc_edge])
#             route_temp.loc[:(loc_edge-1), 'DistTravel']=route_temp['Distance'][:loc_edge].values
        else:
            pass_per_new=SIM_STEPS/route_temp.TravelTime[0]
            
        route_temp.loc[loc_edge, 'DistTravel']=(route_temp['Distance'][loc_edge]-route_temp['DistTravel'][loc_edge])*pass_per_new
        pass_per_dist=route_temp.loc[loc_edge, 'DistTravel']/route_temp.loc[loc_edge, 'Distance']
        
        move_pos=route_temp.geometry[loc_edge].interpolate(pass_per_dist, normalized=True)
        
        # check passing through which O/D
        temp=route_temp.path_od[:loc_edge,]
        self.pass_od=list(temp[~temp.isna()])
        
        # revise the route edge table
        pass_edge=list(route_temp.iloc[0:(loc_edge+1)]['edge_id'])
        route_temp=route_temp.iloc[loc_edge:].reset_index(drop=True)
        route_temp.loc[0, 'TravelTime']=route_temp.TravelTime[0]*(1-pass_per_new)
        
        self.pos=[move_pos.x, move_pos.y]
#         self.edge_pass_per=pass_per_new
        self.edge_pass_per=pass_per_dist
        self.path_info['path_edge']=route_temp
        self.path_info['total_time']=sum(self.path_info['path_edge'].TravelTime)
        self.path_info['pass_edge']=pass_edge
        
        
def replace_nan(obj):
    if isinstance(obj, float) and math.isnan(obj):
        return None
    if isinstance(obj, dict):
        return {k: replace_nan(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [replace_nan(v) for v in obj]
    return obj

In [ ]:
# # check the availability of fixed-route
# def haversine_distance(lon1, lat1, lon2, lat2):
#     R = 6371000  # Earth radius in meters

#     lon1 = np.radians(lon1)
#     lat1 = np.radians(lat1)
#     lon2 = np.radians(lon2)
#     lat2 = np.radians(lat2)

#     dlon = lon2 - lon1
#     dlat = lat2 - lat1

#     a = (
#         np.sin(dlat / 2) ** 2
#         + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
#     )

#     c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

#     return R * c


# def find_available_routes(pax, bus_stop, max_walk_distance=500):
#     df = bus_stop.copy()

#     o_lon, o_lat = pax['origin']
#     d_lon, d_lat = pax['dest']

#     df['origin_distance'] = haversine_distance(
#         o_lon,
#         o_lat,
#         df['PositionLon'].values,
#         df['PositionLat'].values
#     )

#     df['dest_distance'] = haversine_distance(
#         d_lon,
#         d_lat,
#         df['PositionLon'].values,
#         df['PositionLat'].values
#     )


#     origin_stops = df[
#         df['origin_distance'] <= max_walk_distance
#     ].copy()

#     # Candidate alighting stops
#     dest_stops = df[
#         df['dest_distance'] <= max_walk_distance
#     ].copy()


#     feasible = origin_stops.merge(
#         dest_stops,
#         on=[
#             'RouteUID',
#             'SubRouteUID',
#             'Direction'
#         ],
#         suffixes=('_O', '_D')
#     )

#     feasible = feasible[
#         feasible['StopSequence_O']
#         <
#         feasible['StopSequence_D']
#     ].copy()

#     feasible['n_stops'] = (
#         feasible['StopSequence_D']
#         - feasible['StopSequence_O']
#     )


#     result = feasible[
#         [
#             'RouteUID',
#             'SubRouteUID',
#             'Direction',

#             'StopUID_O',
#             'StopSequence_O',
#             'PositionLat_O',
#             'PositionLon_O',
#             'origin_distance_O',

#             'StopUID_D',
#             'StopSequence_D',
#             'PositionLat_D',
#             'PositionLon_D',
#             'dest_distance_D',

#             'n_stops'
#         ]
#     ].copy()

#     # Rename distances for clarity
#     result = result.rename(
#         columns={
#             'origin_distance_O': 'walk_distance_O',
#             'dest_distance_D': 'walk_distance_D'
#         }
#     )


#     result['total_walk_distance'] = (
#         result['walk_distance_O']
#         + result['walk_distance_D']
#     )

#     result = result.sort_values(
#         [
#             'total_walk_distance',
#             'n_stops'
#         ]
#     ).reset_index(drop=True)

#     result=result.merge(bus_timetable, left_on=['SubRouteUID','Direction','StopUID_O'], right_on=['SubRouteUID','Direction','StopUID'])
#     return result

# Simulation based on Historical Data

In [ ]:
# ODT OD data (predicted by demand model)
# drt_od=pd.read_csv('E:/On-demand Transport Service/Analysis/6_Agent-based Modelling/A_OD Analysis (TW Smartcard)/Yilan City/ODdata/ILA BUFFER_500 DRT OD.csv', engine='pyarrow')
# si_demand=drt_od.sort_values(by='Hour').reset_index(drop=True)
# si_demand['Trips_sim']=si_demand.PredTrips/3600*30

drt_od=pd.read_csv('E:/On-demand Transport Service/Analysis/6_Agent-based Modelling/A_OD Analysis (TW Smartcard)/Yilan City/ML model/optimized_results/new_predictions_500.csv', engine='pyarrow')
drt_od["GridID_O"] = "ILA_" + drt_od["GridID_O"].astype(str).str.zfill(4)
drt_od["GridID_D"] = "ILA_" + drt_od["GridID_D"].astype(str).str.zfill(4)
si_demand=drt_od.sort_values(by='Hour').reset_index(drop=True)
si_demand['Trips_sim']=si_demand.predicted_trips/3600*30

pred_pax_list = []
for tt in tqdm(si_demand.Hour.unique()):
    si_demand_temp = si_demand[si_demand['Hour'] == tt].reset_index(drop=True).copy()

    for i in range(120):
        temp = si_demand_temp.loc[:, ['GridID_O','GridID_D','Trips_sim']].copy()

        temp['Time'] = f"{tt}:{i // 2}:{(i % 2) * 30}"
        temp['Trips'] = np.random.poisson(lam=si_demand_temp['Trips_sim'].values, size=len(si_demand_temp))

        temp = temp[temp["Trips"] != 0].copy()
        temp['Time'] = pd.to_datetime(temp['Time'], format='%H:%M:%S').dt.strftime('%H:%M:%S')
        temp['TimeStep']=i+(tt)*120

        pred_pax_list.append(temp)
    
pred_pax=pd.concat(pred_pax_list, ignore_index=True)

temp=gpd.GeoSeries(pred_pax.merge(drt_grid, left_on='GridID_O', right_on='GridID').geometry, crs='EPSG:4326').sample_points(size=1)
pred_pax['X_O']=temp.x
pred_pax['Y_O']=temp.y
temp=gpd.GeoSeries(pred_pax.merge(drt_grid, left_on='GridID_D', right_on='GridID').geometry, crs='EPSG:4326').sample_points(size=1)
pred_pax['X_D']=temp.x
pred_pax['Y_D']=temp.y

print(len(pred_pax))
sum(si_demand.predicted_trips)

In [ ]:
# TEST_ID=1
# pred_pax.to_csv('Data/simulation/ILA_'+str(TEST_ID)+'_BUFFER_500.csv', index=False)

In [ ]:
# pd.DataFrame(pd.Series([x[0:2] for x in pred_pax.Time]).value_counts())

temp=pred_pax.copy()
temp['Hour']=[x[0:2] for x in temp.Time]
temp=temp.groupby('Hour').agg(count=("Hour", "count")).reset_index()
temp['count_per']=temp['count']/sum(temp['count'])
temp

In [ ]:
# demand settings
TEST_ID=1
ope_type='BUFFER_200'
pred_pax=pd.read_csv('Data/simulation/ILA_'+str(TEST_ID)+'_'+ope_type+'.csv')

# vehicles=[AgentVehicle(id=0, seat_capacity=18),
#           AgentVehicle(id=1, seat_capacity=18),
#           AgentVehicle(id=2, seat_capacity=18),
#           AgentVehicle(id=3, seat_capacity=18),
#           AgentVehicle(id=4, seat_capacity=18),
#           AgentVehicle(id=5, seat_capacity=18)]

num_veh=4
vehicles=[AgentVehicle(id=i, seat_capacity=4) for i in range(num_veh)]

passengers = [] 
history = []

In [ ]:
# TT=721

# t=TT

# print(TT)

# pred_pax_temp=pred_pax[pred_pax.TimeStep==t].reset_index(drop=True)

# for PP in range(len(pred_pax_temp)):
#     origin=[float(pred_pax_temp.X_O[PP]), float(pred_pax_temp.Y_O[PP])]
#     dest=[float(pred_pax_temp.X_D[PP]), float(pred_pax_temp.Y_D[PP])]

#     pax_num=1
#     pax_new={'id':len(passengers), 'origin':origin, 'dest':dest, 'status':'waiting', 'pax_num': pax_num}

#     # --- INSERTION HEURISTIC ---
#     # find the Best Vehicle AND the Best Position in its queue

#     best_v = None
#     best_route = None
#     min_added_cost = float('inf')

#     check_capacity=0
#     for v in vehicles:
#         if sum([d['pax_num'] for d in passengers if d["id"] in v.assign_pax])+pax_new['pax_num']>v.seat_capacity:
#             check_capacity=check_capacity+1
#     if check_capacity==len(vehicles):
#         pax_new['status']='reject'
#         break

#     # Apply the best decision found
#     temp=find_best_vehicle(pax_new, vehicles, t/(60/SIM_STEPS))
#     best_v=vehicles[temp['veh_id']]
#     best_v.waypoints=temp['result']['route'] # Update the plan!
#     best_v.assign_pax.append(len(passengers))
#     best_v.path_info=temp['result']['edge_info']
#     best_v.path_info['path_edge']=get_route_edge(best_v.waypoints, temp['result']['edge_info']['path_edge_all'])
#     pax_new['vehicle']=best_v.id
    
#     passengers.append(pax_new)
    
#     # update arrival time
#     for x in temp['result']['edge_info']['predict_arr']:
#         if x[0]=='O':
#             temp_id=int(re.search(r'\d+', x).group())
#             passengers[temp_id]['expected_pick']=temp['result']['edge_info']['predict_arr']['O_'+str(temp_id)]
#         if x[0]=='D':
#             temp_id=int(re.search(r'\d+', x).group())
#             passengers[temp_id]['expected_drop']=temp['result']['edge_info']['predict_arr']['D_'+str(temp_id)]

    


# # MOVE AGENTS
# for v in vehicles:
#     v.move()
#     if len(v.path_info)!=0 or len(v.pass_od)!=0:
#         all_pax=v.assign_pax.copy()
#         for pid in all_pax:
#             p=passengers[pid]
#             if p['status'] == 'waiting':
#                 if 'O_'+str(pid) in v.pass_od:
#                     p['status'] = 'picked_up'
#                     v.waypoints.pop('O_'+str(pid))
#             if p['status'] == 'picked_up':
#                 if 'D_'+str(pid) in v.pass_od:
#                     p['status'] = 'drop_off'
#                     v.assign_pax.remove(pid)
#                     v.waypoints.pop('D_'+str(pid))
                    
# TT=TT+1

# print('waypoints:')
# for v in vehicles:
#     print(v.waypoints)

# print('\nVehicles Posistion:')
# for v in vehicles:
#     print(v.pos)

In [ ]:
# VEHID=3

# path_geo=gpd.GeoDataFrame(vehicles[VEHID].path_info['path_edge'].merge(edges_sf))
# m=folium.Map(location=[path_geo.geometry.unary_union.centroid.y, path_geo.geometry.unary_union.centroid.x], zoom_start=14, tiles='CartoDB positron_nolabels')

# for i in vehicles[VEHID].assign_pax:
#     temp=passengers[i]['origin']
#     folium.Marker(location=[temp[1], temp[0]], popup='Pickup '+str(i), icon=folium.Icon(color='red')).add_to(m)

#     temp=passengers[i]['dest']
#     folium.Marker(location=[temp[1], temp[0]], popup='Dropoff '+str(i), icon=folium.Icon(color='blue')).add_to(m)

# folium.Marker(location=[vehicles[VEHID].pos[1], vehicles[VEHID].pos[0]], popup='Vehicle', icon=folium.Icon(color='green')).add_to(m)

# folium.GeoJson(path_geo).add_to(m)
# folium.LayerControl().add_to(m)

# print(vehicles[VEHID].path_info['path_time'])
# m

In [ ]:
for t in range(min(drt_od.Hour.unique())*120, (max(drt_od.Hour.unique())+1)*120):
    pred_pax_temp=pred_pax[pred_pax.TimeStep==t].reset_index(drop=True)

    for PP in range(len(pred_pax_temp)):
        origin=[float(pred_pax_temp.X_O[PP]), float(pred_pax_temp.Y_O[PP])]
        dest=[float(pred_pax_temp.X_D[PP]), float(pred_pax_temp.Y_D[PP])]
        
        pax_num=1
        pax_new={'id':len(passengers), 'origin':origin, 'dest':dest, 'status':'waiting', 'pax_num': pax_num}

        # --- INSERTION HEURISTIC ---
        # find the Best Vehicle AND the Best Position in its queue

        best_v = None
        best_route = None
        min_added_cost = float('inf')

        print('Find best route for '+str(PP)+' in Time: '+str(t))
        feasible_vehicles=[]
        for v in vehicles:
            assigned_pax = sum(d['pax_num'] for d in passengers if d["id"] in v.assign_pax)
            if assigned_pax + pax_new['pax_num'] <= v.seat_capacity:
                feasible_vehicles.append(v)
            
        if len(feasible_vehicles) == 0:
            pax_new['status']='reject'
            print("Reject")
        else:
            # Apply the best decision found
            temp=find_best_vehicle(pax_new, feasible_vehicles, t/2)
            best_v=feasible_vehicles[temp['veh_id']]
            best_v.waypoints=temp['result']['route'] # Update the plan!
            best_v.assign_pax.append(pax_new['id'])
            best_v.path_info=temp['result']['edge_info']
            best_v.path_info['path_edge']=get_route_edge(best_v.waypoints, temp['result']['edge_info']['path_edge_all'])
            pax_new['vehicle']=best_v.id

        passengers.append(pax_new)

    
    # MOVE AGENTS
    for v in vehicles:
        v.move()
        if len(v.path_info)!=0 or len(v.pass_od)!=0:
            all_pax=v.assign_pax.copy()
            for pid in all_pax:
                p=passengers[pid]
                if p['status'] == 'waiting':
                    if 'O_'+str(pid) in v.pass_od:
                        p['status'] = 'picked_up'
                        v.waypoints.pop('O_'+str(pid))
                if p['status'] == 'picked_up':
                    if 'D_'+str(pid) in v.pass_od:
                        p['status'] = 'drop_off'
                        v.assign_pax.remove(pid)
                        v.waypoints.pop('D_'+str(pid))
        

    print('Time: '+str(t))

    temp1=copy.deepcopy(passengers)
    temp2=copy.deepcopy(vehicles)

    # RECORD FRAME
    frame = {
        'time': t,
        'vehicles': [list(v.pos) for v in temp2],
        'vehicles_pax': [{v.id: v.assign_pax} for v in temp2],
        'passenger': temp1,
        'path_edge': [list(v.path_info['path_edge']['edge_id']) if len(v.path_info)!=0 else [] for v in temp2],
        'pass_edge': [list(v.path_info['pass_edge']) if len(v.path_info)!=0 else [] for v in temp2],
        'edge_pass_per': [v.edge_pass_per if len(v.path_info)!=0 else None for v in temp2]
    }
    
    history.append(frame)

In [ ]:
with open('result/ILA_'+ope_type+'_'+str(num_veh)+'.json', 'w', encoding='utf-8') as json_file:
    json.dump(replace_nan(history), json_file, indent=4)

# Simulation (by operational model and fleet size)

In [ ]:
for num_veh in list(range(5,1,-1)):
    vehicles=[AgentVehicle(id=i, seat_capacity=4) for i in range(num_veh)]

    passengers = [] 
    history = []

    for t in range(min(drt_od.Hour.unique())*120, (max(drt_od.Hour.unique())+1)*120):
        pred_pax_temp=pred_pax[pred_pax.TimeStep==t].reset_index(drop=True)

        for PP in range(len(pred_pax_temp)):
            origin=[float(pred_pax_temp.X_O[PP]), float(pred_pax_temp.Y_O[PP])]
            dest=[float(pred_pax_temp.X_D[PP]), float(pred_pax_temp.Y_D[PP])]

            pax_num=1
            pax_new={'id':len(passengers), 'origin':origin, 'dest':dest, 'status':'waiting', 'pax_num': pax_num}

            # --- INSERTION HEURISTIC ---
            # find the Best Vehicle AND the Best Position in its queue

            best_v = None
            best_route = None
            min_added_cost = float('inf')

            print('Find best route for '+str(PP)+' in Time: '+str(t))
            feasible_vehicles=[]
            for v in vehicles:
                assigned_pax = sum(d['pax_num'] for d in passengers if d["id"] in v.assign_pax)
                if assigned_pax + pax_new['pax_num'] <= v.seat_capacity:
                    feasible_vehicles.append(v)

            if len(feasible_vehicles) == 0:
                pax_new['status']='reject'
                print("Reject")
            else:
                # Apply the best decision found
                temp=find_best_vehicle(pax_new, feasible_vehicles, t/2)
                best_v=feasible_vehicles[temp['veh_id']]
                best_v.waypoints=temp['result']['route'] # Update the plan!
                best_v.assign_pax.append(pax_new['id'])
                best_v.path_info=temp['result']['edge_info']
                best_v.path_info['path_edge']=get_route_edge(best_v.waypoints, temp['result']['edge_info']['path_edge_all'])
                pax_new['vehicle']=best_v.id

            passengers.append(pax_new)

            
            # update arrival time
            for x in temp['result']['edge_info']['predict_arr']:
                if x[0]=='O':
                    temp_id=int(re.search(r'\d+', x).group())
                    passengers[temp_id]['expected_pick']=temp['result']['edge_info']['predict_arr']['O_'+str(temp_id)]
                if x[0]=='D':
                    temp_id=int(re.search(r'\d+', x).group())
                    passengers[temp_id]['expected_drop']=temp['result']['edge_info']['predict_arr']['D_'+str(temp_id)]



        # MOVE AGENTS
        for v in vehicles:
            v.move()
            if len(v.path_info)!=0 or len(v.pass_od)!=0:
                all_pax=v.assign_pax.copy()
                for pid in all_pax:
                    p=passengers[pid]
                    if p['status'] == 'waiting':
                        if 'O_'+str(pid) in v.pass_od:
                            p['status'] = 'picked_up'
                            v.waypoints.pop('O_'+str(pid))
                    if p['status'] == 'picked_up':
                        if 'D_'+str(pid) in v.pass_od:
                            p['status'] = 'drop_off'
                            v.assign_pax.remove(pid)
                            v.waypoints.pop('D_'+str(pid))


        print('Time: '+str(t))

        temp1=copy.deepcopy(passengers)
        temp2=copy.deepcopy(vehicles)

        # RECORD FRAME
        frame = {
            'time': t,
            'vehicles': [list(v.pos) for v in temp2],
            'vehicles_pax': [{v.id: v.assign_pax} for v in temp2],
            'passenger': temp1,
            'path_edge': [list(v.path_info['path_edge']['edge_id']) if len(v.path_info)!=0 else [] for v in temp2],
            'pass_edge': [list(v.path_info['pass_edge']) if len(v.path_info)!=0 else [] for v in temp2],
            'edge_pass_per': [v.edge_pass_per if len(v.path_info)!=0 else None for v in temp2]
        }

        history.append(frame)

    with open('result/ILA_'+ope_type+'_'+str(num_veh)+'.json', 'w', encoding='utf-8') as json_file:
        json.dump(replace_nan(history), json_file, indent=4)

In [ ]:
# def simulation_test(num_veh_all, CON, TEST_ID, ope_type, alpha, beta):
#     pred_pax=pd.read_csv('Data/simulation/'+CON+'_'+str(TEST_ID)+'/'+CON+'_'+ope_type+'.csv')

#     for num_veh in num_veh_all:
#         vehicles=[AgentVehicle(id=i, seat_capacity=18) for i in range(num_veh)]

#         passengers = [] 
#         history = []

#         for t in range(min(odt_od.Trip_Date_Hours.unique())*120, (max(odt_od.Trip_Date_Hours.unique())+1)*120):
#             pred_pax_temp=pred_pax[pred_pax.TimeStep==t].reset_index(drop=True)

#             for PP in range(len(pred_pax_temp)):
#                 origin=[float(pred_pax_temp.X_O[PP]), float(pred_pax_temp.Y_O[PP])]
#                 dest=[float(pred_pax_temp.X_D[PP]), float(pred_pax_temp.Y_D[PP])]

#                 pax_num=1
#                 pax_new={'id':len(passengers), 'origin':origin, 'dest':dest, 'status':'waiting', 'pax_num': pax_num}

#                 # --- INSERTION HEURISTIC ---
#                 # find the Best Vehicle AND the Best Position in its queue

#                 best_v = None
#                 best_route = None
#                 min_added_cost = float('inf')

#                 print('Find best route for '+str(PP)+' in Time: '+str(t))
#                 check_capacity=0
#                 for v in vehicles:
#                     if sum([d['pax_num'] for d in passengers if d["id"] in v.assign_pax])+pax_new['pax_num']>v.seat_capacity:
#                         check_capacity=check_capacity+1
#                 if check_capacity==len(vehicles):
#                     pax_new['status']='reject'
#                     break

#                 # Apply the best decision found
#                 temp=find_best_vehicle(pax_new, vehicles, t/(60/SIM_STEPS), alpha=ALPHA, beta=BETA)
#                 best_v=vehicles[temp['veh_id']]
#                 best_v.waypoints=temp['result']['route'] # Update the plan!
#                 best_v.assign_pax.append(len(passengers))
#                 best_v.path_info=temp['result']['edge_info']
#                 best_v.path_info['path_edge']=get_route_edge(best_v.waypoints, temp['result']['edge_info']['path_edge_all'])
#                 pax_new['vehicle']=best_v.id

#                 passengers.append(pax_new)

#                 # update arrival time
#                 for x in temp['result']['edge_info']['predict_arr']:
#                     if x[0]=='O':
#                         temp_id=int(re.search(r'\d+', x).group())
#                         passengers[temp_id]['expected_pick']=temp['result']['edge_info']['predict_arr']['O_'+str(temp_id)]
#                     if x[0]=='D':
#                         temp_id=int(re.search(r'\d+', x).group())
#                         passengers[temp_id]['expected_drop']=temp['result']['edge_info']['predict_arr']['D_'+str(temp_id)]



#             # MOVE AGENTS
#             for v in vehicles:
#                 v.move()
#                 if len(v.path_info)!=0 or len(v.pass_od)!=0:
#                     all_pax=v.assign_pax.copy()
#                     for pid in all_pax:
#                         p=passengers[pid]
#                         if p['status'] == 'waiting':
#                             if 'O_'+str(pid) in v.pass_od:
#                                 p['status'] = 'picked_up'
#                                 v.waypoints.pop('O_'+str(pid))
#                         if p['status'] == 'picked_up':
#                             if 'D_'+str(pid) in v.pass_od:
#                                 p['status'] = 'drop_off'
#                                 v.assign_pax.remove(pid)
#                                 v.waypoints.pop('D_'+str(pid))


#             print('Time: '+str(t))

#             temp1=copy.deepcopy(passengers)
#             temp2=copy.deepcopy(vehicles)

#             # RECORD FRAME
#             frame = {
#                 'time': t,
#                 'vehicles': [list(v.pos) for v in temp2],
#                 'vehicles_pax': [{v.id: v.assign_pax} for v in temp2],
#                 'passenger': temp1,
#                 'path_edge': [list(v.path_info['path_edge']['edge_id']) if len(v.path_info)!=0 else [] for v in temp2],
#                 'pass_edge': [list(v.path_info['pass_edge']) if len(v.path_info)!=0 else [] for v in temp2],
#                 'edge_pass_per': [v.edge_pass_per if len(v.path_info)!=0 else None for v in temp2]
#             }

#             history.append(frame)
        
#         os.makedirs('result/alpha_'+str(ALPHA)+'_beta_'+str(BETA), exist_ok=True)
#         with open('result/alpha_'+str(ALPHA)+'_beta_'+str(BETA)+'/'+CON+'_'+ope_type+'_'+str(num_veh)+'.json', 'w', encoding='utf-8') as json_file:
#             json.dump(replace_nan(history), json_file, indent=4)